# Import data into the knowledge graph

This notebook keeps the original step-by-step import style: download data, use for loops to select records, add triples, and save the graph.


In [ ]:
%pip install rdflib


## Import Windows ATT&CK techniques


In [ ]:
from urllib.request import urlopen
import json

attack_url = "https://raw.githubusercontent.com/mitre-attack/attack-stix-data/master/enterprise-attack/enterprise-attack.json"
with urlopen(attack_url) as response:
    att_obj = json.load(response)["objects"]

def attack_id(item):
    for ref in item.get("external_references", []):
        if ref.get("source_name") == "mitre-attack":
            return ref.get("external_id", "Unknown")
    return "Unknown"

windows_tech = []
for obj in att_obj:
    if (obj.get("type") == "attack-pattern"
            and "Windows" in obj.get("x_mitre_platforms", [])
            and not obj.get("revoked", False)
            and not obj.get("deprecated", False)):
        windows_tech.append(obj)

print(len(windows_tech))
for technique in windows_tech[:10]:
    print(f"{attack_id(technique)} - {technique['name']}")


## Select five techniques and add their triples


In [ ]:
selected_techniques = []
selected_ids = ["T1566.001", "T1059.001", "T1059.003", "T1053.005", "T1047"]

for technique in windows_tech:
    if attack_id(technique) in selected_ids:
        selected_techniques.append(technique)

from rdflib import Graph, Namespace, RDF, RDFS, URIRef, Literal
g = Graph()
CTI = Namespace("https://github.com/Timoseee/windows-cyber-threat-kg/ontology#")

for technique in selected_techniques:
    technique_id = attack_id(technique)
    technique_uri = URIRef(CTI[technique_id])
    g.add((technique_uri, RDF.type, CTI.AttackTechnique))
    g.add((technique_uri, RDFS.label, Literal(f"{technique['name']} ({technique_id})")))
    g.add((technique_uri, CTI.applicableToPlatform, CTI.Windows))

    for phase in technique.get("kill_chain_phases", []):
        tactic_name = phase["phase_name"].replace("-", " ").title()
        tactic_uri = URIRef(CTI[tactic_name.replace(" ", "")])
        g.add((tactic_uri, RDF.type, CTI.AttackTactic))
        g.add((tactic_uri, RDFS.label, Literal(tactic_name)))
        g.add((technique_uri, CTI.belongsToTactic, tactic_uri))

print("Selected techniques:", len(selected_techniques))


## Import threat actors and their relationships


In [ ]:
select_id = []
for technique in selected_techniques:
    select_id.append(technique["id"])

relationship = []
for item in att_obj:
    if (item.get("type") == "relationship"
            and item.get("relationship_type") == "uses"
            and item.get("target_ref") in select_id):
        relationship.append(item)

def _find_item(stix_id):
    for item in att_obj:
        if item.get("id") == stix_id:
            return item
    return None

nw_actor_relation = []
for relation in relationship:
    source_item = _find_item(relation.get("source_ref"))
    if source_item is not None and source_item.get("type") == "intrusion-set":
        nw_actor_relation.append(relation)

for relation in nw_actor_relation:
    source_item = _find_item(relation.get("source_ref"))
    target_item = _find_item(relation.get("target_ref"))
    actor_id = attack_id(source_item)
    technique_id = attack_id(target_item)

    g.add((URIRef(CTI[actor_id]), RDF.type, CTI.ThreatActor))
    g.add((URIRef(CTI[actor_id]), RDFS.label, Literal(f"{source_item['name']} ({actor_id})")))
    g.add((URIRef(CTI[actor_id]), CTI.usesTechnique, URIRef(CTI[technique_id])))

print("Actor relationships:", len(nw_actor_relation))


## Import selected CVEs from NVD


In [ ]:
cve_ids = ["CVE-2024-30051", "CVE-2024-30078", "CVE-2024-38063", "CVE-2024-38080"]
record = []

for cve_id in cve_ids:
    url = "https://services.nvd.nist.gov/rest/json/cves/2.0" + f"?cveId={cve_id}"
    with urlopen(url) as response:
        nvd_data = json.load(response)
    cve = nvd_data["vulnerabilities"][0]["cve"]
    record.append(cve)

print("Downloaded CVE records:", len(record))


## Add CVE, Windows product, CWE, and severity triples


In [ ]:
for cve in record:
    cve_id = cve["id"]
    cve_uri = CTI[cve_id.replace("-", "_")]
    g.add((cve_uri, RDF.type, CTI.Vulnerability))
    g.add((cve_uri, RDFS.label, Literal(cve_id)))

    for config in cve["configurations"]:
        for node in config["nodes"]:
            for cpe_match in node.get("cpeMatch", []):
                parts = cpe_match["criteria"].split(":")
                vendor = parts[3]
                product = parts[4]
                version = parts[5]
                if vendor != "microsoft" or not product.startswith("windows"):
                    continue

                product_name = f"{product} {version}"
                product_uri = CTI[f"{product}_{version}"]
                g.add((product_uri, RDF.type, CTI.OperatingSystem))
                g.add((product_uri, RDFS.label, Literal(product_name)))
                g.add((product_uri, CTI.belongsToPlatform, CTI.Windows))
                g.add((cve_uri, CTI.affectsProduct, product_uri))

    for weakness in cve.get("weaknesses", []):
        cwe_id = weakness["description"][0]["value"]
        if cwe_id.startswith("CWE-"):
            weakness_uri = CTI[cwe_id.replace("-", "_")]
            g.add((weakness_uri, RDF.type, CTI.Weakness))
            g.add((weakness_uri, RDFS.label, Literal(cwe_id)))
            g.add((cve_uri, CTI.hasWeakness, weakness_uri))

    cvss = cve["metrics"]["cvssMetricV31"][0]["cvssData"]
    severity = cvss["baseSeverity"]
    severity_uri = CTI[severity.title() + "Severity"]
    g.add((severity_uri, RDF.type, CTI.SeverityLevel))
    g.add((severity_uri, RDFS.label, Literal(severity + " severity")))
    g.add((cve_uri, CTI.hasSeverity, severity_uri))

print("Graph now contains:", len(g), "triples")


## Save the graph


In [ ]:
save_place = r"C:\\Users\\lenovo\\OneDrive\\Desktop\\windows-cyber-threat-kg-imported.ttl"
g.serialize(destination=save_place, format="turtle")
print(save_place)
